In [ ]:
!pip install numpy==1.26.4 --force-reinstall --quiet


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install --force-reinstall opencv-python-headless==4.8.1.78
!pip install -U ultralytics


In [ ]:
!pip install ultralytics==8.3.209

from ultralytics import YOLO
import torch, shutil

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
zip_path = "/kaggle/input/obb-dataset-co4/custom_merged_obb.zip"
extract_path = "/kaggle/input/obb-dataset-co4/custom_merged_obb"

if os.path.exists(zip_path):
    print("Extracting dataset...")
    shutil.unpack_archive(zip_path, extract_path)
else:
    print("Dataset already extracted.")

print("Dataset path:", extract_path)

In [ ]:


yaml_path = "/kaggle/input/obb-dataset-co4/custom_merged_obb/data.yaml"

# Just print and visually confirm
!cat {yaml_path}

# Optionally check that the train/valid folders exist
print("\nChecking folders:")
!ls /kaggle/input/obb-dataset-co3/custom_merged_obb/images
!ls /kaggle/input/obb-dataset-co3/custom_merged_obb/images/valid | head -n 5


In [ ]:
!pip install ultralytics==8.3.209 opencv-python matplotlib --quiet

In [ ]:
from ultralytics import YOLO
print("Ultralytics version check passed")


In [ ]:
from ultralytics.data.utils import check_det_dataset

print("Checking dataset integrity...")
dataset_info = check_det_dataset(yaml_path)
print("Dataset verified successfully!")
print(f"Train: {dataset_info['train']}")
print(f"Val:   {dataset_info.get('val', 'N/A')}")
print(f"Classes: {dataset_info['names']}")

In [ ]:
import os

base = "/kaggle/input/obb-dataset-co4/custom_merged_obb"
for sub in ["images/train", "images/valid", "labels/train", "labels/valid"]:
    path = os.path.join(base, sub)
    print(f"\n Checking {path}")
    print("Count:", len(os.listdir(path)))
    bad = [f for f in os.listdir(path) if os.path.getsize(os.path.join(path, f)) == 0]
    if bad:
        print("Empty files:", bad[:5])
    else:
        print("No empty files found")


In [ ]:
import shutil, os

src = "/kaggle/input/obb-dataset-co4/custom_merged_obb"
dst = "/kaggle/working/custom_merged_obb"

# Remove the old dataset if it exists
if os.path.exists(dst):
    shutil.rmtree(dst)
    print("Old dataset removed from /kaggle/working/")

In [ ]:
import shutil, os

src = "/kaggle/input/obb-dataset-co4/custom_merged_obb"
dst = "/kaggle/working/custom_merged_obb"

# copy entire dataset tree (takes ~1-2 min)
if not os.path.exists(dst):
    shutil.copytree(src, dst)
    print("Dataset copied to /kaggle/working/")
else:
    print("Dataset already exists in /kaggle/working/")


In [ ]:
# clean the copy in /kaggle/working/
def remove_empty_labels(base_path):
    count_removed = 0
    for sub in ["labels/train", "labels/valid"]:
        folder = os.path.join(base_path, sub)
        for f in os.listdir(folder):
            path = os.path.join(folder, f)
            if os.path.getsize(path) == 0:
                os.remove(path)
                img_name = f.replace(".txt", ".jpg")
                for img_dir in ["images/train", "images/valid"]:
                    img_path = os.path.join(base_path, img_dir, img_name)
                    if os.path.exists(img_path):
                        os.remove(img_path)
                count_removed += 1
    print(f"Removed {count_removed} empty label files and their images.")

remove_empty_labels("/kaggle/working/custom_merged_obb")


In [ ]:

import sys, types, os

# Disable external loggers
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
os.environ["RAY_AIR_NEW_OUTPUT"] = "0"

# Create dummy ray modules so any 'import ray.train._internal.session' resolves safely
ray = types.ModuleType("ray")
train = types.ModuleType("train")
_internal = types.ModuleType("_internal")
session = types.ModuleType("session")
session._get_session = lambda: None   # <- the problematic call returns None harmlessly
_internal.session = session
train._internal = _internal
ray.train = train
sys.modules["ray"] = ray
sys.modules["ray.train"] = train
sys.modules["ray.train._internal"] = _internal
sys.modules["ray.train._internal.session"] = session

print("Ray hard-blocked & W&B disabled")


In [ ]:
yaml_path = "/kaggle/working/custom_merged_obb/data.yaml"
with open(yaml_path, "r") as f:
    txt = f.read()
txt = txt.replace("/kaggle/input/obb-dataset-co4/custom_merged_obb", "/kaggle/working/custom_merged_obb")
with open(yaml_path, "w") as f:
    f.write(txt)
print("data.yaml now points to /kaggle/working/custom_merged_obb")


In [ ]:
import ultralytics
print(ultralytics.__version__)


In [ ]:
!find /kaggle/working/custom_merged_obb -name "*.cache" -delete


In [ ]:
import os

base = "/kaggle/working/custom_merged_obb"
for subset in ["train", "valid", "test"]:
    img_dir = os.path.join(base, "images", subset)
    lbl_dir = os.path.join(base, "labels", subset)
    if not os.path.exists(img_dir): 
        continue
    imgs = {os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))}
    lbls = {os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if f.endswith('.txt')}
    missing_imgs = lbls - imgs
    missing_lbls = imgs - lbls
    print(f"{subset}: {len(missing_imgs)} labels missing images, {len(missing_lbls)} images missing labels.")


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s-obb.pt")
results = model.train(
    data="/kaggle/working/custom_merged_obb/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    amp=False,
    device=0,
    workers=0,
    cache=False,
    project="obb_run",
    name="train_final_fixed",
    save_period=5,
    patience=20,
    lr0=0.01,
    augment=True
)



In [ ]:
import os
os.listdir("/kaggle/working/obb_run/train_final_fixed/weights")


In [ ]:
import os, shutil

final_path = "/kaggle/working/final_weights"
os.makedirs(final_path, exist_ok=True)

src_folder = "/kaggle/working/obb_run/train_final_fixed/weights"

for w in ["best.pt", "last.pt"]:
    src = os.path.join(src_folder, w)
    if os.path.exists(src):
        shutil.copy(src, final_path)
        print(f"Copied {w} → {final_path}")

print("Check /kaggle/working/final_weights for saved models.")


In [ ]:
import shutil

# Path to your training results
src_folder = "/kaggle/working/obb_run"
zip_path = "/kaggle/working/train_final_fixed_all.zip"

# Create ZIP archive
shutil.make_archive(zip_path.replace(".zip", ""), "zip", src_folder)
print("Folder zipped successfully:", zip_path)


In [ ]:
# =============================================
# Auto-generate YOLO validation plots (mAP, PR, F1, Confusion Matrix)
# =============================================
from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd
import glob
import cv2
import os

# Path to best weights and dataset yaml
best_model = "/kaggle/working/obb_run/train_final_fixed/weights/best.pt"
data_yaml = "/kaggle/working/custom_merged_obb/data.yaml"

# Validate and generate all plots
model = YOLO(best_model)
metrics = model.val(data=data_yaml, save=True)
print(f"Validation complete — results saved to: {metrics.save_dir}")

# Plot losses and mAP curves from CSV
csv_path = os.path.join(metrics.save_dir, "results.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    plt.figure()
    plt.plot(df["epoch"], df["train/box_loss"], label="Box Loss")
    plt.plot(df["epoch"], df["train/cls_loss"], label="Cls Loss")
    plt.plot(df["epoch"], df["train/dfl_loss"], label="DFL Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curves")
    plt.legend(); plt.grid(True); plt.show()

    plt.figure()
    plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5")
    plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
    plt.xlabel("Epoch"); plt.ylabel("mAP"); plt.title("Validation mAP Progress")
    plt.legend(); plt.grid(True); plt.show()
else:
    print("results.csv not found; skipping curve plots")

# Display all YOLO-generated images (curves, confusion matrices)
for img_path in glob.glob(os.path.join(metrics.save_dir, "*.png")):
    print("🖼️", os.path.basename(img_path))
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load training log
df = pd.read_csv("/kaggle/working/obb_run/train_final_fixed/results.csv")

# Plot loss curves
plt.figure()
plt.plot(df["epoch"], df["train/box_loss"], label="Box Loss")
plt.plot(df["epoch"], df["train/cls_loss"], label="Cls Loss")
plt.plot(df["epoch"], df["train/dfl_loss"], label="DFL Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curves")
plt.legend()
plt.grid(True)
plt.show()

# Plot mAP curves
plt.figure()
plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5")
plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation mAP Progress")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from ultralytics import YOLO

# Load the best model
model = YOLO("/kaggle/working/obb_run/train_final_fixed/weights/best.pt")

# Run validation on the same validation split defined in your data.yaml
metrics = model.val(data="/kaggle/working/custom_merged_obb/data.yaml")

# Print summary
print(metrics)


In [ ]:
# Visualize a few validation images with predicted rotated boxes
model.predict(
    source="/kaggle/working/custom_merged_obb/images/valid",
    conf=0.5,
    save=True,
    project="obb_run",
    name="val_predictions"
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = r"E:\path\to\obb_run\train_final_fixed\results.csv"
df = pd.read_csv(csv_path)

plt.figure(figsize=(8, 5))
plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5")
plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation mAP Progress")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(df["epoch"], df["train/box_loss"], label="Box Loss")
plt.plot(df["epoch"], df["train/cls_loss"], label="Cls Loss")
plt.plot(df["epoch"], df["train/dfl_loss"], label="DFL Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curves")
plt.legend()
plt.grid(True)
plt.show()
